# 22481 · Lesson 9 — Python Image Analysis Toolbox
**6 November 2026 | Reusable functions + optional practice**

**Two equally valid starting points:** **A — Practise first** with the synthetic example in this notebook, or **B — Investigate now** using your own images in *My Phantom Investigation*. Move freely between the two.

### What this toolbox is (and is not)
This is a **starting point, not a complete analysis protocol**. You may copy, adapt, combine and extend the functions. Their explanations describe **what the code does**, its inputs and important assumptions—not **which scientific questions to ask**, where to place ROIs, which thresholds to choose or what conclusions to draw. Those decisions belong to your group.

**Scientific questions should guide your analysis—not the other way around.** You are not assessed on the amount of original Python code you write. You are responsible for understanding your measurements, checking calibration, evaluating uncertainty and presenting reproducible evidence.

**How to use:** Run the setup and function-definition cells first. The cells marked **OPTIONAL PRACTICE** demonstrate individual tools on **synthetic, non-clinical data**; you may skip them. Work on your actual investigation in the companion notebook. The tools were adapted from the course's earlier DICOM exploration, display/processing and ROI exercises, with additional measurement helpers.

## 1 · Setup and DICOM helpers
These functions load DICOM images, apply stored rescale information and display key metadata. **Read the notes in each function**, especially about image geometry. A folder containing multiple DICOM series requires selecting the correct `SeriesInstanceUID`.

**Scope:** The loader assumes ordinary single-frame grayscale DICOM images. Colour ultrasound, enhanced multiframe DICOM and other special formats may require a different loader. Check the data before analysing it.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import pydicom
print("Toolbox functions ready to define.")

In [ ]:
def _read_dicom(path):
    """Read one DICOM image file; fail clearly if it has no pixel data."""
    ds = pydicom.dcmread(str(path))
    if not hasattr(ds, "PixelData"):
        raise ValueError(f"No image pixel data in {path}")
    return ds

def _pixels(ds):
    """Return rescaled pixel values as floats (not necessarily calibrated HU)."""
    a = np.asarray(ds.pixel_array).astype(float)
    # DICOM can store pixel values with a linear slope/intercept conversion.
    slope = float(getattr(ds, "RescaleSlope", 1))
    intercept = float(getattr(ds, "RescaleIntercept", 0))
    return a * slope + intercept

def load_dicom(path, series_uid=None):
    """Load a DICOM file or a single series from a directory.

    Parameters
    ----------
    path : str or Path
        A single image file or directory containing DICOM files.
    series_uid : str or None
        Select a specific series when the directory contains several.

    Returns
    -------
    image_or_volume : ndarray
        2D for one file; 3D (slice, row, column) for a directory.
    first_dataset : pydicom Dataset
        Header of the first image in the selected series.
    datasets : list
        DICOM datasets in the same order as the returned slices.

    Assumptions: single-frame grayscale images with matching dimensions.
    Check orientation, slice spacing, pixel spacing and units before measuring.
    """
    path = Path(path)
    if path.is_file():
        ds = _read_dicom(path)
        return _pixels(ds), ds, [ds]
    if not path.is_dir():
        raise FileNotFoundError(path)
    groups = {}
    # Scan metadata first so that unrelated series are not mixed together.
    for p in sorted(path.rglob("*")):
        if not p.is_file() or p.name.startswith("."):
            continue
        try:
            h = pydicom.dcmread(str(p), stop_before_pixels=True)
            if not hasattr(h, "Rows"):
                continue
            uid = str(getattr(h, "SeriesInstanceUID", "unknown"))
            groups.setdefault(uid, []).append(p)
        except Exception:
            continue
    if not groups:
        raise ValueError("No readable DICOM image files found")
    print("Available series:")
    for uid, paths in groups.items():
        print(f"  {uid} : {len(paths)} files")
    if series_uid is None and len(groups) > 1:
        raise ValueError("Multiple series: set SERIES_UID to one listed above")
    uid = series_uid or next(iter(groups))
    if uid not in groups:
        raise ValueError("SERIES_UID not found")
    dsets = [_read_dicom(p) for p in groups[uid]]
    def sort_key(ds):
        """Sort slices by physical position if available; otherwise instance number."""
        try:
            orient = np.asarray(ds.ImageOrientationPatient, float)
            normal = np.cross(orient[:3], orient[3:])
            return float(np.dot(np.asarray(ds.ImagePositionPatient, float), normal))
        except Exception:
            return float(getattr(ds, "InstanceNumber", 0))
    dsets.sort(key=sort_key)
    arrs = [_pixels(ds) for ds in dsets]
    if len({a.shape for a in arrs}) != 1:
        raise ValueError("Different image shapes within series")
    return np.stack(arrs, axis=0), dsets[0], dsets

def pixel_spacing_mm(ds):
    """Return (row_spacing, column_spacing) in mm/pixel, or None if absent."""
    if hasattr(ds, "PixelSpacing"):
        return tuple(float(v) for v in ds.PixelSpacing)
    return None

def show_metadata(ds, arr):
    """Print image dimensions and metadata needed to judge measurement validity."""
    print("Array shape:", arr.shape)
    for field in ["Modality", "SeriesDescription", "PixelSpacing", "SliceThickness", "SpacingBetweenSlices", "PhotometricInterpretation", "RescaleSlope", "RescaleIntercept"]:
        print(f"{field}: {getattr(ds, field, 'not provided')}")
    print("WARNING: confirm image geometry, pixel calibration, and orientation before physical measurements.")

# PixelSpacing is (row, column) mm/pixel, NOT (x, y).
# Slice thickness is not necessarily the distance between adjacent slice centres.
# Do not combine different series without checking geometry and scaling.
# This basic loader is not designed for colour ultrasound or multiframe data.

## 2 · Image display and synthetic demonstration
The demonstration phantom is for practice only. To use your own data, load it in the investigation notebook and select a meaningful slice.

In [ ]:
def make_demo():
    """Create a synthetic 2D image with simple structures and added variation.

    This image is ONLY for learning the functions. Intensities and spacing
    are arbitrary; they are not CT HU or measurements of a real phantom.
    """
    yy, xx = np.mgrid[:160, :200]
    img = np.full((160, 200), 18.0)
    # Add structures of different shapes/intensities to test analysis choices.
    img[((xx-62)/25)**2 + ((yy-80)/38)**2 < 1] = 65
    img[((xx-62)/10)**2 + ((yy-80)/14)**2 < 1] = 125
    img[((xx-140)/24)**2 + ((yy-83)/27)**2 < 1] = 42
    rng = np.random.default_rng(22481)
    return img + rng.normal(0, 3, img.shape)

In [ ]:
def show_image(img, spacing=None, vmin=None, vmax=None, title="Image"):
    """Display a 2D image with optional physical axes.

    spacing : (row_mm, column_mm) or None
        If omitted, axes show pixels. If supplied, axes show mm.
    vmin, vmax : optional display limits
        Change image appearance only; do not change underlying pixel values.
    Returns (figure, axes) for further annotations or saving.
    """
    fig, ax = plt.subplots(figsize=(7, 5))
    if spacing is None:
        ax.imshow(img, cmap="gray", origin="upper", vmin=vmin, vmax=vmax)
        ax.set(xlabel="Column (pixels)", ylabel="Row (pixels)")
    else:
        dy, dx = spacing  # DICOM stores row spacing before column spacing.
        ax.imshow(img, cmap="gray", origin="upper", vmin=vmin, vmax=vmax,
                  extent=[0, img.shape[1]*dx, img.shape[0]*dy, 0])
        ax.set(xlabel="Horizontal distance (mm)", ylabel="Vertical distance (mm)")
    ax.set_title(title)
    return fig, ax

In [ ]:
# OPTIONAL PRACTICE: use a deliberately simple synthetic phantom (not your own data).
practice_image = make_demo()
practice_spacing = (0.5, 0.5)  # row, column mm/pixel; DEMO ONLY
fig, ax = show_image(practice_image, practice_spacing, title="Synthetic practice phantom")
plt.show()


## 3 · Physical dimensions and line profiles
**Question examples:** How wide is the structure? Are there internal layers? Verify calibration and choose meaningful endpoints. Pixel coordinates use `(column, row)`.

In [ ]:
def distance_mm(p1_xy, p2_xy, spacing):
    """Distance in mm between two pixel positions (x=column, y=row).

    spacing must be verified (row_mm, column_mm). Suitable for a 2D
    image with orthogonal in-plane pixel axes; not a 3D distance.
    """
    if spacing is None:
        raise ValueError("Physical pixel spacing not verified")
    dy, dx = spacing
    # Convert horizontal and vertical pixel differences separately to mm.
    return float(np.hypot((p2_xy[0]-p1_xy[0])*dx, (p2_xy[1]-p1_xy[1])*dy))

def line_profile(img, p1_xy, p2_xy, spacing=None, samples=300):
    """Sample intensities along a line between two (column, row) positions.

    Returns (distance_along_line, intensity_values, distance_unit).
    Uses nearest-neighbour sampling: repeated pixels and stair-step edges
    are possible. Choose endpoints based on your scientific question.
    """
    x = np.linspace(p1_xy[0], p2_xy[0], samples)
    y = np.linspace(p1_xy[1], p2_xy[1], samples)
    # Round sampling locations to nearest existing pixels (no interpolation).
    xi = np.clip(np.rint(x).astype(int), 0, img.shape[1]-1)
    yi = np.clip(np.rint(y).astype(int), 0, img.shape[0]-1)
    values = img[yi, xi]
    if spacing is None:
        dist = np.linspace(0, np.hypot(x[-1]-x[0], y[-1]-y[0]), samples)
        unit = "pixels"
    else:
        dist = np.linspace(0, distance_mm(p1_xy, p2_xy, spacing), samples)
        unit = "mm"
    return dist, values, unit

In [ ]:
# OPTIONAL PRACTICE: choose different endpoints and see how the profile changes.
P1, P2 = (25, 80), (100, 80)  # (column, row) pixel coordinates, DEMO ONLY
fig, ax = show_image(practice_image, title="Practice line")
ax.plot([P1[0], P2[0]], [P1[1], P2[1]], "r-")
plt.show()
d, vals, unit = line_profile(practice_image, P1, P2, practice_spacing)
plt.plot(d, vals)
plt.xlabel(f"Distance ({unit})"); plt.ylabel("Synthetic intensity")
plt.show()
print("Endpoint distance:", distance_mm(P1, P2, practice_spacing), "mm")


## 4 · ROIs and quantitative contrast
ROI results depend on location, size, heterogeneity and image scaling. The within-ROI standard deviation is **not automatically image noise**. Do not compute SNR/CNR without justifying a suitable noise estimate.

In [ ]:
def roi_values(img, roi):
    """Return pixel intensities inside a rectangular ROI.

    roi = (x, y, width, height), all in pixels; x is column, y is row.
    The rectangle must lie completely inside the 2D image.
    """
    x, y, w, h = map(int, roi)
    if w <= 0 or h <= 0 or x < 0 or y < 0 or x+w > img.shape[1] or y+h > img.shape[0]:
        raise ValueError(f"ROI outside image: {roi}")
    # Array indexing is [row, column], the reverse of (x, y) notation.
    return np.asarray(img[y:y+h, x:x+w], float).ravel()

def roi_summary(img, roi):
    """Summarise an ROI with count, mean, median and within-ROI SD.

    Output intensity units are those of the input image. The standard
    deviation reflects ALL variation within the ROI (including structure,
    artefacts and noise); it is not automatically a noise estimate.
    """
    a = roi_values(img, roi)
    return {"n_pixels": len(a), "mean": float(a.mean()),
            "std_within_roi": float(a.std(ddof=1)) if len(a)>1 else np.nan,
            "median": float(np.median(a))}

In [ ]:
# OPTIONAL PRACTICE: move ROIs to different regions and compare means.
roi_a, roi_b = (50, 62, 12, 12), (126, 67, 12, 12)  # x,y,width,height; DEMO ONLY
fig, ax = show_image(practice_image, title="Practice ROIs")
for rect, color in [(roi_a, "red"), (roi_b, "cyan")]:
    x,y,w,h = rect
    ax.add_patch(Rectangle((x,y),w,h,fill=False,edgecolor=color,linewidth=2))
plt.show()
print("A", roi_summary(practice_image, roi_a))
print("B", roi_summary(practice_image, roi_b))
print("Mean difference:", roi_summary(practice_image,roi_a)["mean"]-roi_summary(practice_image,roi_b)["mean"])


## 5 · Optional segmentation and smoothing
A threshold-based mask is one possible way to estimate a region's area; it is **not automatic object identification**. The example below uses a synthetic image and a demonstration threshold, not a recommended threshold for your phantom.

Filtering and thresholds can change apparent dimensions and boundary visibility. Compare with the original image, check physical calibration and report processing decisions. Segmentation and SNR/CNR are **optional**, not required checkboxes.

In [ ]:
# OPTIONAL PRACTICE: thresholding classifies pixels by intensity, not material identity.
# The threshold is an analytical choice; test its effect on the measured area.
threshold = 50  # DEMO ONLY
mask = practice_image > threshold
fig, axes = plt.subplots(1,2,figsize=(9,4))
axes[0].imshow(practice_image,cmap="gray"); axes[0].set_title("Original")
axes[1].imshow(mask,cmap="gray"); axes[1].set_title("Threshold mask")
plt.tight_layout(); plt.show()
print("Total mask area (not necessarily one object):",mask.sum()*np.prod(practice_spacing),"mm²")
try:
    from scipy.ndimage import gaussian_filter
    smoothed = gaussian_filter(practice_image, sigma=1.5)
    fig,axes=plt.subplots(1,2,figsize=(9,4))
    axes[0].imshow(practice_image,cmap="gray"); axes[0].set_title("Original")
    axes[1].imshow(smoothed,cmap="gray"); axes[1].set_title("Gaussian smoothing")
    plt.tight_layout(); plt.show()
except ImportError:
    print("Install scipy to try optional smoothing.")


## 6 · Working with your own images
In **My Phantom Investigation.ipynb**, start with a question. Copy only the functions you need. For DICOM: `volume, ds, slices = load_dicom(DATA_PATH, SERIES_UID)`; inspect `show_metadata(ds, volume)` and `pixel_spacing_mm(ds)`. For a 3D stack choose a slice with `image = volume[index]`. **Important:** a 2D colour/multiframe image may not fit this pattern. Confirm spatial scale before physical measurements.

Use the report guide to compare X-ray, CT, ultrasound and MRI. A single line profile or ROI is rarely sufficient to establish object identity. Document *question → method → result → interpretation/limitations*.

**Scientific caution:** CT values may be in HU only when the DICOM rescale is correct; MR and ultrasound pixel values are usually not directly comparable between acquisitions. Projection X-ray sizes can be magnified and overlapping objects can hide one another. Do not equate within-ROI variation with random noise. For volume measurement, check slice geometry rather than using `SliceThickness` blindly.

**Before reporting:** Are your chosen ROIs/lines/thresholds scientifically justified? Could a different selection change the conclusion? Show original data alongside processed figures when relevant.